In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
CUDA version: 12.1
PyTorch version: 2.5.1+cu121
GPU device: NVIDIA GeForce RTX 4060 Laptop GPU
Number of GPUs: 1


In [1]:
import logging

from dotenv import load_dotenv
from langchain_core.documents import Document

from rag.ingestion.llama_parse_processor import process_document
from rag.ingestion.document_fetcher import DocumentFetcher
from rag.ingestion.document_tracker import DocumentTracker
from utils.api_utils import DefineEdgeFundamentalsAPI
from rag.ingestion.vector_store import QdrantManager
from utils.data_helpers import (
    initialize_metadata_data,
    initialize_stock_data,
    symbol_to_fincode,
)

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

c:\Users\Anant\Downloads\embed-documents-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-08 20:19:16,197 - utils.data_helpers - INFO - Initializing stock data mappings...
2026-02-08 20:19:16,210 - utils.api_utils - INFO - Retrieved active equity and sub listing data from cache
2026-02-08 20:19:16,230 - utils.data_helpers - INFO - Stock data initialized: 5517 stocks, 5501 symbols
2026-02-08 20:19:16,231 - utils.data_helpers - INFO - Initializing metadata mappings (20 years of data)...
2026-02-08 20:19:16,254 - utils.api_utils - INFO - Retrieved meta data file for 2006-02-13 to 2026-02-08 from cache
2026-02-08 20:19:16,276 - utils.data_helpers - INFO - Metadata initialized: 22109 items, 4064 unique fincodes


### Get Stocks from Nifty 50 Index

In [2]:
api = DefineEdgeFundamentalsAPI()

groups = await api.get_predefined_groups()

all_group_names = groups.keys()
for gname in all_group_names:
    if gname == "Nifty 50 Index":
        nifty_50_stocks = groups[gname]

symbol_to_fincode_map = {}
for symbol in nifty_50_stocks:
    symbol_to_fincode_map[symbol] = symbol_to_fincode(symbol)

fincodes = list(symbol_to_fincode_map.values())

2026-02-08 20:19:29,507 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/groups/0 "HTTP/1.1 200 OK"


In [3]:
fetcher = DocumentFetcher()
doc_tracker = DocumentTracker()
qdrant_manager = QdrantManager() 

2026-02-08 20:19:39,388 - rag.ingestion.document_tracker - INFO - Initialized DocumentTracker with database at c:\Users\Anant\Downloads\embed-documents-main\embed-documents-main\rag\data\embedded_docs.db
2026-02-08 20:19:40,484 - rag.ingestion.document_tracker - INFO - Initialized DocumentTracker with database at c:\Users\Anant\Downloads\embed-documents-main\embed-documents-main\rag\data\embedded_docs.db
2026-02-08 20:19:40,486 - rag.ingestion.vector_store - INFO - Initialized QdrantManager with collection 'company_files'


In [4]:
for fincode in fincodes:
    # fetch documents for the fincode
    try:
        docs = await fetcher.get_available_documents(
            fincode=fincode,
        )
        for doc in docs:
            logger.info(
                f"Document: {doc.filename} ({doc.category}) - {doc.document_date}"
            )
    except Exception as e:
        logger.error(f"Error fetching documents for fincode {fincode}: {e}")

    try:
        # check which documents already exist in vector store
        exists = await doc_tracker.check_documents_exist([doc.filename for doc in docs])
    except Exception as e:
        logger.error(f"Error checking document existence for fincode {fincode}: {e}")
        continue

    # process fincode documents
    try:
        processed_docs: list[Document] = []

        for doc in docs:
            if not exists[doc.filename]:
                try:
                    logger.info(
                        f"Document {doc.filename} does not exist in vector store. Proceeding to fetch and save."
                    )
                    pdf = await fetcher.get_pdf_doc_stream(doc.filename, doc.category)

                    file_size = len(pdf.stream.getbuffer()) / 1024 / 1024

                    logger.info(f"Document size: {file_size:,} bytes ({file_size:.2f} MB)")

                    processed_docs.extend(await process_document(pdf, file_type="pdf"))
                except Exception as e:
                    logger.error(f"Error processing document {doc.filename}: {e}")
                    continue
    except Exception as e:
        logger.error(f"Error processing documents for fincode {fincode}: {e}")
        continue

    # Select documents that need to be embedded
    try:
        docs_to_embed: list[Document] = []
        for doc in processed_docs:
            if not exists[doc.metadata.get("source", "")]:
                docs_to_embed.append(doc)

        for doc in docs_to_embed:
            logger.info(f"Document to embed: {doc.metadata.get('source', '')}")
    except Exception as e:
        logger.error(f"Error selecting documents to embed for fincode {fincode}: {e}")
        if docs_to_embed.empty():
            logger.info(f"No new documents to embed for fincode {fincode}. Skipping.")
            continue
    
    try:
        if isinstance(docs_to_embed, list) and len(docs_to_embed) > 0:
            result = await qdrant_manager.embed_documents(docs_to_embed)
            logger.info(f"Cost: ${result['estimated_cost_usd']}")
    except Exception as e:
        logger.error(f"Error embedding documents for fincode {fincode}: {e}")
        continue

2026-02-08 20:19:48,473 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=112599, category=None, date_range=2024-02-09 to 2026-02-08
2026-02-08 20:19:48,524 - utils.api_utils - INFO - Retrieved meta data file for 2024-02-09 to 2026-02-08 from cache
2026-02-08 20:19:48,527 - rag.ingestion.document_fetcher - INFO - Found 11 documents matching filters
2026-02-08 20:19:48,529 - __main__ - INFO - Document: 5a639c02-968a-4206-a4e6-39c9b3219bd9.pdf (concall) - 2025-11-10
2026-02-08 20:19:48,530 - __main__ - INFO - Document: 8de70047-ce26-4a8e-b11f-7621bc26af13.pdf (investor-presentation) - 2025-11-04
2026-02-08 20:19:48,530 - __main__ - INFO - Document: e15fe661-864e-453a-905f-9a8a26e3583c.pdf (concall) - 2025-08-05
2026-02-08 20:19:48,531 - __main__ - INFO - Document: 51e2e786-7289-44df-95f4-781f4fe5935a.pdf (investor-presentation) - 2025-07-31
2026-02-08 20:19:48,532 - __main__ - INFO - Document: 1547dc88-edf8-4ba5-8526-184d835ce26a.pdf (concall) - 2025-05-07
202

Started parsing the file under job_id 3dcb5077-fef9-4ac0-a093-974c8fdbf33a


2026-02-08 20:19:54,874 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/3dcb5077-fef9-4ac0-a093-974c8fdbf33a "HTTP/1.1 200 OK"
2026-02-08 20:19:57,241 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/3dcb5077-fef9-4ac0-a093-974c8fdbf33a "HTTP/1.1 200 OK"
2026-02-08 20:20:00,620 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/3dcb5077-fef9-4ac0-a093-974c8fdbf33a "HTTP/1.1 200 OK"
2026-02-08 20:20:05,002 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/3dcb5077-fef9-4ac0-a093-974c8fdbf33a "HTTP/1.1 200 OK"
2026-02-08 20:20:05,587 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/3dcb5077-fef9-4ac0-a093-974c8fdbf33a/result/markdown "HTTP/1.1 200 OK"
2026-02-08 20:20:05,591 - __main__ - INFO - Document 8de70047-ce26-4a8e-b11f-7621bc26af13.pdf does not exist in vector store. Proceeding to fetch and save.
2026-02-08 20:20:0

Started parsing the file under job_id 65c9225f-a7d0-4e2a-98fe-6b449f36e38b


2026-02-08 20:20:12,674 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:15,251 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:18,837 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:23,232 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:29,267 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:35,178 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/65c9225f-a7d0-4e2a-98fe-6b449f36e38b "HTTP/1.1 200 OK"
2026-02-08 20:20:41,114 - ht

...

Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmp_qapy5ix.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}


2026-02-08 23:03:25,861 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=annual-report&attachmentName=53cec073-5278-4dcd-9066-2fd101023af5.pdf "HTTP/1.1 200 OK"
2026-02-08 23:03:27,051 - __main__ - INFO - Document size: 11.93527603149414 bytes (11.94 MB)
2026-02-08 23:03:31,336 - httpx - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 402 Payment Required"
2026-02-08 23:03:31,363 - __main__ - INFO - Document 64445460-0bf5-44b1-adb3-59d8b8416860.pdf does not exist in vector store. Proceeding to fetch and save.


Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmpv9n3egms.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}


2026-02-08 23:03:33,140 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=64445460-0bf5-44b1-adb3-59d8b8416860.pdf "HTTP/1.1 200 OK"
2026-02-08 23:03:33,252 - __main__ - INFO - Document size: 0.9686603546142578 bytes (0.97 MB)
2026-02-08 23:03:37,030 - httpx - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 402 Payment Required"


Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmprkq1y4xj.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}
